# Code mẫu — Tìm kiếm và suy luận

Một thuật toán tìm kiếm phải tối ưu đúng đại lượng đề hỏi, và không được vứt bỏ những trạng thái có thể dẫn tới đáp án tốt. Ba đồ thị nhỏ dưới đây cho thấy khi nào BFS cho đáp án chính xác, khi nào A* cần một heuristic không ước lượng quá tay, và khi nào beam search chấp nhận đánh đổi đáp án để giảm số ứng viên phải giữ.

## Ví dụ 1: Đi qua ít cửa nhất trên một bản đồ

Bản đồ gồm các phòng, mỗi cạnh là một lần đi qua cửa, và mọi cửa có chi phí như nhau, bằng 1. Từ S tới G có đường `S, A, G` qua hai cửa, và đường `S, B, C, G` qua ba cửa. BFS phải tìm ra đường hai cửa. Ta kiểm lại bằng cách liệt kê mọi đường đi đơn, một cách hoàn toàn độc lập với hàng đợi của BFS.

In [ ]:
from collections import deque

graph = {"S": ["B", "A"], "A": ["G"], "B": ["C"], "C": ["G"], "G": []}
queue, seen = deque([["S"]]), {"S"}
answer = None
while queue:
    path = queue.popleft()
    if path[-1] == "G":
        answer = path
        break
    for following in graph[path[-1]]:
        if following not in seen:
            seen.add(following)
            queue.append(path + [following])

def all_paths(path):
    if path[-1] == "G":
        return [path]
    results = []
    for following in graph[path[-1]]:
        if following not in path:
            results.extend(all_paths(path + [following]))
    return results

oracle = all_paths(["S"])
assert len(answer) - 1 == min(len(path) - 1 for path in oracle) == 2
print("BFS:", answer, "số cửa:", len(answer) - 1)
print("độ dài mọi đường:", sorted(len(path) - 1 for path in oracle))

BFS: ['S', 'A', 'G'] số cửa: 2
độ dài mọi đường: [2, 3]


Dù B đứng trước A trong danh sách kề của S, BFS vẫn tìm ra đường hai cửa. BFS duyệt theo từng lớp khoảng cách, nên đường đầu tiên chạm tới G luôn là đường qua ít cạnh nhất.

Kết luận này chỉ đúng khi mọi cửa tốn như nhau. Nếu đề hỏi thời gian, và hai cửa tốn tổng 20 giây trong khi ba cửa chỉ tốn 5 giây, thì đường ít cửa nhất không còn là đường nhanh nhất. Khi đó phải dùng Dijkstra hoặc A*.

## Ví dụ 2: Heuristic ước lượng quá tay có thể đưa A* vào đường đắt

Robot đi từ S tới G. Nhánh qua B tốn $1+9=10$ giây. Nhánh qua A tốn $2+2=4$ giây. A* xếp ưu tiên theo $g+h$, trong đó g là chi phí đã đi, còn h là heuristic, ước lượng chi phí còn lại. Ta chạy cùng một A* với ba heuristic: toàn số 0, đúng bằng chi phí còn lại thật, và một bản cố tình ước lượng quá tay ở A.

In [ ]:
import heapq
import math

graph = {"S": [("B", 1), ("A", 2)], "A": [("G", 2)],
         "B": [("G", 9)], "G": []}

def astar(heuristic):
    heap = [(heuristic["S"], 0, "S", ("S",))]
    best = {"S": 0}
    opened = []
    while heap:
        _, cost, node, path = heapq.heappop(heap)
        if cost != best[node]:
            continue
        opened.append(node)
        if node == "G":
            return cost, path, opened
        for following, edge_cost in graph[node]:
            candidate = cost + edge_cost
            if candidate < best.get(following, math.inf):
                best[following] = candidate
                heapq.heappush(heap, (candidate + heuristic[following],
                                     candidate, following, path + (following,)))
    return math.inf, (), opened

def exact_remaining(node, used=()):
    if node == "G":
        return 0
    return min((cost + exact_remaining(following, used + (node,))
                for following, cost in graph[node] if following not in used),
               default=math.inf)

truth = {node: exact_remaining(node) for node in graph}
good = {"S": 4, "A": 2, "B": 9, "G": 0}
bad = {"S": 0, "A": 20, "B": 0, "G": 0}
assert all(0 <= good[node] <= truth[node] for node in graph)
assert all(good[node] <= cost + good[following]
           for node in graph for following, cost in graph[node])
for name, h in [("h=0", dict.fromkeys(graph, 0)), ("lower bound", good), ("phóng đại", bad)]:
    cost, path, opened = astar(h)
    print(name, "giây:", cost, "đường:", path, "mở:", opened)
assert astar(good)[0] == truth["S"] == 4
assert astar(bad)[0] == 10
print("h sai tại:", [node for node in graph if bad[node] > truth[node]])

h=0 giây: 4 đường: ('S', 'A', 'G') mở: ['S', 'B', 'A', 'G']
lower bound giây: 4 đường: ('S', 'A', 'G') mở: ['S', 'A', 'G']
phóng đại giây: 10 đường: ('S', 'B', 'G') mở: ['S', 'B', 'G']
h sai tại: ['A']


Với h toàn 0, A* chính là Dijkstra. Nó tìm đúng đường 4 giây, nhưng phải mở cả B. Với heuristic bằng đúng chi phí còn lại, A có ưu tiên $2+2=4$, còn B có $1+9=10$, nên A* đi thẳng qua A và không đụng tới B. Hai dòng assert đầu kiểm heuristic tốt không bao giờ ước lượng vượt chi phí thật, và nhất quán dọc theo mọi cạnh.

Bản ước lượng quá tay gán cho A giá trị 20, trong khi thật ra chỉ còn 2. Ưu tiên của A thành 22, và G qua B được lấy ra trước với giá 10. A* trả về đường đắt gấp hai lần rưỡi. Dừng đúng lúc lấy G ra khỏi heap cũng không cứu được, khi heuristic đã sai. Lưu ý thêm cách viết: code cho phép cập nhật khi tìm được đường rẻ hơn tới một đỉnh, và bỏ qua những mục cũ còn nằm trong heap, thay vì khóa đỉnh ngay lúc đẩy vào heap.

## Ví dụ 3: Beam rộng hơn cứu được một nhánh khởi đầu yếu

Một model chấm điểm từng bước ghép đoạn. Đi từ S sang A được 6 điểm, từ S sang B được 5. Sau A thì tới G ngay, không thêm điểm. Sau B thì đi qua C, được thêm 10 điểm, rồi tới G. Mục tiêu là đường hợp lệ có tổng điểm lớn nhất. Bảng điểm này tự đặt, không phải chất lượng ngôn ngữ của model nào.

In [ ]:
graph = {"S": [("A", 6), ("B", 5)], "A": [("G", 0)],
         "B": [("C", 10)], "C": [("G", 0)], "G": []}

def exhaustive(node="S", path=("S",), score=0):
    if node == "G":
        return [(score, path)]
    results = []
    for following, extra in graph[node]:
        if following not in path:
            results.extend(exhaustive(following, path + (following,), score + extra))
    return results

def beam(width):
    frontier, completed = [(0, ("S",))], []
    kept_by_level = []
    for _ in range(3):
        candidates = []
        for score, path in frontier:
            for following, extra in graph[path[-1]]:
                if following not in path:
                    item = (score + extra, path + (following,))
                    if following == "G":
                        completed.append(item)
                    else:
                        candidates.append(item)
        frontier = sorted(candidates, reverse=True)[:width]
        kept_by_level.append([path[-1] for score, path in frontier])
        if not frontier:
            break
    return max(completed), kept_by_level

optimum = max(exhaustive())
print("vét cạn:", optimum)
for width in [1, 2]:
    result, trace = beam(width)
    print("beam", width, "kết quả:", result, "nhánh giữ:", trace)
    if width == 2:
        assert result == optimum
assert beam(1)[0][0] == 6 and optimum[0] == 15

vét cạn: (15, ('S', 'B', 'C', 'G'))
beam 1 kết quả: (6, ('S', 'A', 'G')) nhánh giữ: [['A'], []]
beam 2 kết quả: (15, ('S', 'B', 'C', 'G')) nhánh giữ: [['A', 'B'], ['C'], []]


Duyệt hết cho đáp án tối ưu là đường qua B và C, tổng 15 điểm. Beam rộng 1 chỉ giữ A ở tầng đầu, vì 6 lớn hơn 5, và bỏ B vĩnh viễn. Kết quả chỉ được 6 điểm. Beam rộng 2 còn giữ B, nên thấy được 10 điểm thưởng phía sau, và đạt đúng 15.

Beam search cắt nhánh dựa trên điểm *tạm thời*, chứ không có bằng chứng rằng nhánh bị cắt không thể thắng. Điều này khác hẳn với việc cắt một nhánh đã vi phạm ràng buộc của đề, vì nhánh ấy chắc chắn không hợp lệ. Tăng độ rộng lên 2 cứu được bài nhỏ này, nhưng không bảo đảm sẽ tìm ra tối ưu trên những bài lớn hơn.